In [ ]:
%load_ext autoreload
%autoreload 2

import os
from pathlib import Path
import sys

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

from configs import load_yaml
from dataset import FacesDataset
from transforms import get_train_transform, get_test_transform

from train_ebgan import build_models_from_config, train_ebgan_with_schedule, estimate_initial_energy, train_ebgan_core, compute_fid
from copy import deepcopy
from torchvision.utils import make_grid, save_image
from train_ebgan import build_models_from_config, train_ebgan_core, compute_fid

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
batch_size = 32

# Load data
train_df = pd.read_csv("data/train.csv")
test_df = pd.read_csv("data/test.csv")

# Add image paths pointing to processed_64
for df in [train_df, test_df]:
    df["image_path"] = df["id"].apply(
        lambda x: f"data/processed_64/face-{int(x)}.png"
    )

# Create datasets (keep dataset objects so later cells can use them)
train_dataset = FacesDataset(train_df, transform=get_train_transform(64))
test_dataset = FacesDataset(test_df, transform=get_test_transform(64))

# Minimal DataLoaders
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)


In [ ]:
base_cfg = dict(vars(load_yaml('configs/training.yaml')))

In [ ]:
# Experiment configurations (direct training-config overrides)
experiments = [
    {"name": "base"},
    {"name": "k3_g1", "discriminator_steps": 3, "generator_steps": 1},
    {"name": "k5_g2", "discriminator_steps": 5, "generator_steps": 2},
    {"name": "g_relu_e_relu", "generator_activation": "relu", "discriminator_activation": "relu"},
    {"name": "g_gelu_e_leaky", "generator_activation": "gelu", "discriminator_activation": "leakyrelu"},
    {"name": "real70_fake30", "real_ratio_for_discriminator": 0.7},
    {"name": "real30_fake70", "real_ratio_for_discriminator": 0.3},
    {"name": "dropout_10", "generator_dropout": 0.1, "discriminator_dropout": 0.1},
    {"name": "dropout_20", "generator_dropout": 0.2, "discriminator_dropout": 0.2},
    {"name": "no_dgm_norm", "generator_norm": "none"},
    {"name": "no_dem_norm", "discriminator_norm": "none"},
]

print('Loaded', len(experiments), 'experiments')


In [ ]:
ablation_dir = Path('outputs/ablation')
ablation_dir.mkdir(parents=True, exist_ok=True)

# Use one fixed latent batch for all experiments so samples are comparable.
fixed_z = torch.randn(16, int(base_cfg['latent_dim']), 1, 1, device=device)

ablation_results = []
rows_for_csv = []

for exp in experiments:
    exp_name = exp['name']
    cfg = deepcopy(base_cfg)

    # Directly apply experiment overrides (all keys except name)
    overrides = {k: v for k, v in exp.items() if k != 'name'}
    cfg.update(overrides)

    print(f"\n[ABLATION] Training {exp_name} for {cfg['epochs']} epochs ...")

    generator, discriminator = build_models_from_config(cfg)
    states, _, _ = train_ebgan_core(
        generator=generator,
        discriminator=discriminator,
        train_loader=train_loader,
        device=device,
        config=cfg,
        start_epoch=1,
        end_epoch=int(cfg['epochs']),
        eval_loader=None,
    )

    # Compute FID after training and store it in last_state
    last_state = states[-1] if len(states) else {}
    fid_val = compute_fid(
        generator,
        test_loader,
        device,
        latent_dim=int(cfg['latent_dim']),
        num_batches=1,
    )
    last_state['fid'] = float(fid_val)

    # Sample and save fixed grid after training (same latent input for all models)
    generator.eval()
    with torch.no_grad():
        fake_imgs = generator(fixed_z)
        grid = make_grid(fake_imgs, nrow=4, normalize=True, value_range=(-1, 1))

    out_path = ablation_dir / f"{exp_name}.png"
    save_image(grid, out_path)

    ablation_results.append({
        'name': exp_name,
        'sample_path': str(out_path),
        'last_metrics': last_state,
    })

    rows_for_csv.append({
        'name': exp_name,
        'sample_path': str(out_path),
        'g_loss': last_state.get('g_loss'),
        'd_loss': last_state.get('d_loss'),
        'e_real_train': last_state.get('e_real_train'),
        'e_fake_train': last_state.get('e_fake_train'),
        'pt_train': last_state.get('pt_train'),
        'fid': last_state.get('fid'),
    })

    print(f"[ABLATION] Saved sample: {out_path} | fid={last_state.get('fid')}")

# Save CSV summary
csv_path = ablation_dir / 'results.csv'
pd.DataFrame(rows_for_csv).to_csv(csv_path, index=False)

print(f"\nDone. Generated {len(ablation_results)} samples in {ablation_dir}")
print(f"Saved CSV results to: {csv_path}")
ablation_results[:3]